# 05 扩展内容框架

本节为第二章中尚未展开的主题保留入口。代码单元故意保持简短，作为后续扩展 Hermite 插值、PCHIP、B 样条和二维插值的可运行草稿。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


## 三次 Hermite 插值

Hermite 插值同时使用函数值和导数值。在一个区间上，三次 Hermite 形式可以写成四个基函数的组合：

$$
p(t)=h_{00}(t)y_0+h_{10}(t)m_0+h_{01}(t)y_1+h_{11}(t)m_1,
\qquad 0\le t\le 1.
$$

后续可以把这些基函数与 PCHIP 的斜率选择规则联系起来。


In [ ]:
def hermite_basis(t):
    t = np.asarray(t, dtype=float)
    h00 = 2 * t**3 - 3 * t**2 + 1
    h10 = t**3 - 2 * t**2 + t
    h01 = -2 * t**3 + 3 * t**2
    h11 = t**3 - t**2
    return h00, h10, h01, h11

t = np.linspace(0.0, 1.0, 300)
labels = ["h00", "h10", "h01", "h11"]

plt.figure(figsize=(8, 4.5))
for values, label in zip(hermite_basis(t), labels):
    plt.plot(t, values, label=label)
plt.xlabel("t")
plt.ylabel("基函数值")
plt.title("三次 Hermite 基函数")
plt.legend()
plt.tight_layout()
plt.show()


## PCHIP 框架

PCHIP 是一种分段三次 Hermite 方法，通过谨慎选择节点斜率来保持单调性并减少过冲。后续应补充：

* 如何由相邻割线斜率估计节点导数；
* 哪些情况下需要把节点斜率设为零以保持单调性；
* 在单调数据上与自然三次样条进行对比。


## 三次均匀 B 样条框架

B 样条使用具有局部支撑的基函数。三次均匀 B 样条的一个基函数只在有限区间内非零。后续扩展时应重点保留“局部支撑”这一思想。


In [ ]:
def truncated_power_3(x):
    return np.maximum(x, 0.0) ** 3


def cubic_uniform_b_spline_basis(x):
    return (
        truncated_power_3(x)
        - 4 * truncated_power_3(x - 1)
        + 6 * truncated_power_3(x - 2)
        - 4 * truncated_power_3(x - 3)
        + truncated_power_3(x - 4)
    ) / 6.0

x_basis = np.linspace(-1.0, 6.0, 700)

plt.figure(figsize=(8, 4.5))
for shift in range(4):
    plt.plot(x_basis, cubic_uniform_b_spline_basis(x_basis - shift), label=f"B(x-{shift})")
plt.xlabel("x")
plt.ylabel("基函数值")
plt.title("平移后的三次均匀 B 样条基函数")
plt.legend()
plt.tight_layout()
plt.show()


## 二维插值框架

二维插值可以先从两个简单情形开始：

* 矩形单元上的双线性插值；
* 三角形单元上的二维一次 Lagrange 插值。

这两类例子足以把本章连接到网格数据、图像处理和有限元方法。


In [ ]:
def bilinear_unit_square(x, y, z00, z10, z01, z11):
    return (
        (1 - x) * (1 - y) * z00
        + x * (1 - y) * z10
        + (1 - x) * y * z01
        + x * y * z11
    )

u = np.linspace(0.0, 1.0, 80)
v = np.linspace(0.0, 1.0, 80)
U, V = np.meshgrid(u, v)
Z = bilinear_unit_square(U, V, z00=0.0, z10=1.0, z01=0.5, z11=0.2)

plt.figure(figsize=(6, 5))
contours = plt.contourf(U, V, Z, levels=20)
plt.colorbar(contours, label="插值结果")
plt.xlabel("x")
plt.ylabel("y")
plt.title("单位正方形上的双线性插值")
plt.tight_layout()
plt.show()


In [ ]:
def barycentric_coordinates(point, triangle):
    p = np.asarray(point, dtype=float)
    tri = np.asarray(triangle, dtype=float)
    matrix = np.array([
        [tri[0, 0], tri[1, 0], tri[2, 0]],
        [tri[0, 1], tri[1, 1], tri[2, 1]],
        [1.0, 1.0, 1.0],
    ])
    rhs = np.array([p[0], p[1], 1.0])
    return np.linalg.solve(matrix, rhs)

triangle = np.array([[0.0, 0.0], [1.0, 0.0], [0.2, 0.9]])
values = np.array([1.0, 2.0, 0.5])
point = np.array([0.35, 0.25])
lambdas = barycentric_coordinates(point, triangle)
interpolated_value = lambdas @ values

print("重心坐标:", lambdas)
print("插值结果:", interpolated_value)


## 扩展小结

Hermite 插值、PCHIP、B 样条和二维插值都是第二章的后续扩展内容。本节先给出可运行框架，方便之后逐步加深理论推导、算法实现和实验对比，同时不打乱已经完成的主线内容。
